# Коломна · 02 · Карты в стиле презентации

Карты строятся по токенам дизайн-системы «Коломна» (раздел «Карты»): суша `map-land`, застройка `map-building`, вода `map-water`, зелень `map-green`, объект исследования `map-focus` с чёрной обводкой 2 pt, граница — пунктир 3 pt (штрих 6, пробел 4), маршрут `map-route` 4 pt, шкала интенсивности `map-seq-1…5`.
На каждой карте — масштабная линейка, стрелка «С», легенда, граница территории и источник.

Размеры PNG — как в `MapFrame`: **3456×1344** (во всю ширину слайда) или **1664×1344** (половина).

Сначала выполните `01_kolomna_data.ipynb` — этот ноутбук читает его результаты из `kolomna_data/`.

In [ ]:
!pip -q install osmnx==2.0.1

In [ ]:
import os, json, urllib.request
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Polygon as MplPolygon, Rectangle
from matplotlib.lines import Line2D
from shapely.geometry import box, Point

IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/kolomna-identity'   # ← папка с клоном репозитория
else:
    PROJECT_DIR = os.getcwd()   # корень клона репозитория
DATA = os.path.join(PROJECT_DIR, 'kolomna_data')
MAPS = os.path.join(PROJECT_DIR, 'kolomna_maps'); os.makedirs(MAPS, exist_ok=True)
CRS = 32636   # UTM 36N — метры, для линейки и площадей

## Токены и шрифты

Значения скопированы из `tokens.md` дизайн-системы. Шрифты — Prata (заголовки), PT Sans (текст), PT Sans Narrow Bold (метки) из Google Fonts; без сети откатываются на DejaVu.

In [ ]:
T = {  # дизайн-система «Коломна», тема paper
  'ground':'#fefefe', 'text':'#020202', 'text-muted':'#5c5852', 'accent':'#e28723', 'accent-text':'#9a5410',
  'pigment-ink':'#020202', 'pigment-paper':'#fefefe', 'brick':'#936e4f', 'canal':'#8c97bd',
  'map-land':'#f4f1ec', 'map-building':'#d6cfc4', 'map-road':'#fefefe', 'map-water':'#8c97bd',
  'map-green':'#a7b08a', 'map-focus':'#e28723', 'map-boundary':'#020202', 'map-route':'#936e4f',
  'map-seq':['#f8e3c8','#efb877','#e28723','#9a5410','#3d2206'],
}
CATS = [T['accent'], T['pigment-ink'], T['canal'], T['brick']]   # порядок категорий по README

FONT_URLS = {
  'Prata':        'https://github.com/google/fonts/raw/main/ofl/prata/Prata-Regular.ttf',
  'PT Sans':      'https://github.com/google/fonts/raw/main/ofl/ptsans/PT_Sans-Web-Regular.ttf',
  'PT Sans Italic':'https://github.com/google/fonts/raw/main/ofl/ptsans/PT_Sans-Web-Italic.ttf',
  'PT Sans Narrow':'https://github.com/google/fonts/raw/main/ofl/ptsansnarrow/PT_Sans-Narrow-Web-Bold.ttf',
}
FONT_DIR = os.path.join(DATA, 'fonts'); os.makedirs(FONT_DIR, exist_ok=True)
FP = {}
for name, url in FONT_URLS.items():
    fn = os.path.join(FONT_DIR, url.rsplit('/',1)[1])
    try:
        if not os.path.exists(fn): urllib.request.urlretrieve(url, fn)
        font_manager.fontManager.addfont(fn); FP[name] = font_manager.FontProperties(fname=fn)
    except Exception as e:
        print('нет шрифта', name, e); FP[name] = font_manager.FontProperties(family='DejaVu Sans')
mpl.rcParams.update({'font.family': FP['PT Sans'].get_name(), 'axes.facecolor': T['map-land'],
                     'figure.facecolor': T['ground'], 'savefig.facecolor': T['ground']})

## Подложка из OSM

Вода, зелень, застройка и улицы в рамке карты. Скачивается один раз и кэшируется в `kolomna_data/basemap_*.gpkg`.

In [ ]:
import osmnx as ox
ox.settings.use_cache = True; ox.settings.cache_folder = os.path.join(DATA, 'cache', 'osmnx')
VIEW = (30.250, 59.908, 30.335, 59.942)   # lon_min, lat_min, lon_max, lat_max — общая рамка всех карт

def layer(name, tags, kinds=('Polygon','MultiPolygon')):
    fn = os.path.join(DATA, f'basemap_{name}.gpkg')
    if os.path.exists(fn): return gpd.read_file(fn)
    g = ox.features_from_bbox(VIEW, tags)
    g = g[g.geom_type.isin(kinds)].reset_index()[['geometry']]
    g.to_file(fn, driver='GPKG'); return g

water = layer('water', {'natural':'water', 'water':True, 'waterway':'riverbank'}).to_crs(CRS)
green = layer('green', {'leisure':['park','garden'], 'landuse':['grass'], 'natural':['wood']}).to_crs(CRS)
blds  = layer('buildings', {'building':True}).to_crs(CRS)
roads = layer('roads', {'highway':['primary','secondary','tertiary','residential','unclassified','pedestrian','living_street']},
              kinds=('LineString','MultiLineString')).to_crs(CRS)

var  = gpd.read_file(os.path.join(DATA, 'kolomna_boundary_variants.geojson')).to_crs(CRS)
osmp = gpd.read_file(os.path.join(DATA, 'osm_polygons.geojson')).to_crs(CRS)
V = {r.key: r.geometry for r in var.itertuples()}
print(len(water), len(green), len(blds), len(roads))

## Оформление: линейка, стрелка «С», легенда, источник

In [ ]:
PX = {'full': (3456, 1344), 'half': (1664, 1344)}
DPI = 200

def new_map(size='full', bounds=None, title=None, kicker=None):
    w, h = PX[size]
    fig = plt.figure(figsize=(w/DPI, h/DPI), dpi=DPI)
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_axis_off()
    x0, y0, x1, y1 = bounds if bounds is not None else gpd.GeoSeries([box(*VIEW)], crs=4326).to_crs(CRS).total_bounds
    # подгоняем рамку под пропорции листа, центр сохраняем
    cx, cy, rw, rh = (x0+x1)/2, (y0+y1)/2, x1-x0, y1-y0
    if rw/rh < w/h: rw = rh*w/h
    else: rh = rw*h/w
    ax.set_xlim(cx-rw/2, cx+rw/2); ax.set_ylim(cy-rh/2, cy+rh/2)
    ax.add_patch(Rectangle((cx-rw/2, cy-rh/2), rw, rh, color=T['map-land'], zorder=0))
    water.plot(ax=ax, color=T['map-water'], lw=0, zorder=1)
    green.plot(ax=ax, color=T['map-green'], lw=0, zorder=2)
    roads.plot(ax=ax, color=T['map-road'], lw=1.2, zorder=3)
    blds.plot(ax=ax, color=T['map-building'], lw=0, zorder=4)
    if kicker: ax.text(0.015, 0.955, kicker.upper(), transform=ax.transAxes, fontproperties=FP['PT Sans Narrow'],
                       fontsize=11, color=T['text'], va='top', zorder=20)
    if title: ax.text(0.015, 0.92, title, transform=ax.transAxes, fontproperties=FP['Prata'],
                      fontsize=22, color=T['text'], va='top', zorder=20)
    return fig, ax

def boundary(ax, geom, lw=3, color=None, label=None, z=10):
    gpd.GeoSeries([geom], crs=CRS).boundary.plot(ax=ax, color=color or T['map-boundary'], lw=lw,
                                                  linestyle=(0, (6, 4)), zorder=z, label=label)

def scalebar(ax, length=500, loc=(0.015, 0.075)):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    sx = x0 + loc[0]*(x1-x0); sy = y0 + loc[1]*(y1-y0)
    h = (y1-y0)*0.008
    ax.add_patch(Rectangle((sx-(x1-x0)*0.006, sy-h*3), length+(x1-x0)*0.012, h*9.5, color=T['map-road'], zorder=30))
    for i in range(2):
        ax.add_patch(Rectangle((sx+i*length/2, sy), length/2, h, facecolor=T['pigment-ink'] if i==0 else T['pigment-paper'],
                               edgecolor=T['pigment-ink'], lw=1, zorder=31))
    for d in (0, length/2, length):
        ax.text(sx+d, sy+h*2, f'{int(d)}' if d < length else f'{int(d)} м', ha='center', va='bottom',
                fontproperties=FP['PT Sans Narrow'], fontsize=9, color=T['text'], zorder=32)

def north(ax, loc=(0.975, 0.86), size=0.05):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    cx = x0+loc[0]*(x1-x0); cy = y0+loc[1]*(y1-y0); s = size*(y1-y0)
    ax.add_patch(MplPolygon([[cx, cy+s], [cx-s*0.35, cy-s*0.4], [cx, cy-s*0.15]], color=T['pigment-ink'], zorder=31))
    ax.add_patch(MplPolygon([[cx, cy+s], [cx+s*0.35, cy-s*0.4], [cx, cy-s*0.15]], facecolor=T['pigment-paper'],
                            edgecolor=T['pigment-ink'], lw=1, zorder=31))
    ax.text(cx, cy+s*1.15, 'С', ha='center', va='bottom', fontproperties=FP['PT Sans Narrow'], fontsize=14, zorder=32)

def legend(ax, handles, loc='lower right', title=None):
    lg = ax.legend(handles=handles, loc=loc, frameon=True, fancybox=False, framealpha=1, edgecolor=T['pigment-ink'],
                   prop=FP['PT Sans'], fontsize=10, title=title, borderpad=0.8, labelspacing=0.6)
    lg.get_frame().set_linewidth(1); lg.set_zorder(40)
    if title: lg.get_title().set_fontproperties(FP['PT Sans Narrow']); lg.get_title().set_fontsize(10)
    for t in lg.get_texts(): t.set_fontproperties(FP['PT Sans']); t.set_fontsize(10)

def source(ax, text):
    ax.text(0.015, 0.012, text, transform=ax.transAxes, ha='left', va='bottom', fontproperties=FP['PT Sans Italic'],
            fontsize=8, color=T['text-muted'], zorder=40,
            bbox=dict(facecolor=T['ground'], edgecolor='none', pad=2))

def save(fig, name):
    fn = os.path.join(MAPS, f'{name}.png'); fig.savefig(fn, dpi=DPI); plt.close(fig); print(fn); return fn

def zoom(geom, pad=250):
    x0, y0, x1, y1 = geom.bounds; return (x0-pad, y0-pad, x1+pad, y1+pad)

SRC_OSM = 'Карта: команда «Коломна», ИТМО, 2026 · данные © участники OpenStreetMap'

## Карта 1. Четыре Коломны: официальная граница и вернакулярные версии

In [ ]:
fig, ax = new_map('full', bounds=zoom(V['версия 4'], 300), kicker='I · Территория · границы',
                  title='Четыре Коломны')
fills = [('версия 4', T['map-seq'][0]), ('версия 3', T['map-seq'][1]), ('версия 2', T['map-seq'][2]), ('версия 1', T['map-seq'][3])]
# кольца без наложения прозрачности: 1, 2 без 1, 3 без 2, 4 без 3
order = ['версия 1','версия 2','версия 3','версия 4']
for i, (k, c) in enumerate(fills[::-1]):
    g = V[k] if i == 0 else V[k].difference(V[order[i-1]])
    gpd.GeoSeries([g], crs=CRS).plot(ax=ax, color=c, alpha=0.85, lw=0, zorder=5)
boundary(ax, V['версия 2'], lw=3, z=12)
for k, c in fills:
    gpd.GeoSeries([V[k]], crs=CRS).boundary.plot(ax=ax, color=T['pigment-ink'], lw=0.6, zorder=11)
labels = {r.key: f"{r.key.split()[1]}. {r.label} — {r.area_km2:.2f}".replace('.', '§', 1).replace('.', ',').replace('§', '.') + ' км²' for r in var.itertuples()}
h = [mpl.patches.Patch(facecolor=c, alpha=0.85, edgecolor=T['pigment-ink'], lw=0.6, label=labels[k]) for k, c in fills[::-1]]
h.append(Line2D([], [], color=T['map-boundary'], lw=3, linestyle=(0,(6,4)), label='граница МО Коломна'))
legend(ax, h, title='ВЕРСИИ ГРАНИЦ')
scalebar(ax); north(ax)
source(ax, SRC_OSM + ' · версии границ: наш анализ (ноутбук 01)')
save(fig, '01_kolomna_boundaries')

## Карта 2. Объекты культурного наследия (ЕГРОКН)

In [ ]:
okn = gpd.read_file(os.path.join(DATA, 'okn_kolomna_egrokn.geojson')).to_crs(CRS)
fig, ax = new_map('full', bounds=zoom(V['версия 4'], 300), kicker='II · Когнитивный компонент · наследие',
                  title='Памятники — по краям, среда — в центре')
for k, w in [('версия 4', 1), ('версия 3', 1)]:
    gpd.GeoSeries([V[k]], crs=CRS).boundary.plot(ax=ax, color=T['text-muted'], lw=w, zorder=9)
boundary(ax, V['версия 2'])
fed = okn[okn['cat'].str.startswith('Фед')]; reg = okn[okn['cat'].str.startswith('Рег')]
reg.plot(ax=ax, color=T['pigment-ink'], markersize=30, zorder=14)
fed.plot(ax=ax, color=T['map-focus'], edgecolor=T['pigment-ink'], linewidth=2, markersize=60, zorder=15)
n_in = int(okn['версия 2'].sum())
h = [Line2D([], [], marker='o', ls='', markerfacecolor=T['map-focus'], markeredgecolor=T['pigment-ink'], markeredgewidth=2, markersize=9,
            label=f'федерального значения ({len(fed)})'),
     Line2D([], [], marker='o', ls='', color=T['pigment-ink'], markersize=6, label=f'регионального значения ({len(reg)})'),
     Line2D([], [], color=T['map-boundary'], lw=3, linestyle=(0,(6,4)), label=f'МО Коломна — {n_in} объектов с координатами'),
     Line2D([], [], color=T['text-muted'], lw=1, label='версии 3 и 4')]
legend(ax, h, title='ОКН В ЕГРОКН')
scalebar(ax); north(ax)
source(ax, 'Данные: ЕГРОКН (Минкультуры России), выгрузка data-69-structure-16 · ' + SRC_OSM)
save(fig, '02_kolomna_okn')

## Карта 3. Плотность сервиса: тихое ядро и шумные края

Шестиугольники 150 м; считаем магазины, общепит, гостиницы и хостелы из OSM (как в таблице «ядро → кольца»).

In [ ]:
poi = gpd.read_file(os.path.join(DATA, 'osm_poi.geojson')).to_crs(CRS)
serv = poi[poi['serv'] == True]
fig, ax = new_map('full', bounds=zoom(V['версия 4'], 300), kicker='II · Практический компонент · сервис',
                  title='Чем дальше от Крюкова канала, тем тише')
xs, ys = serv.geometry.x.values, serv.geometry.y.values
cmap = mpl.colors.ListedColormap(T['map-seq'])
hb = ax.hexbin(xs, ys, gridsize=(int((ax.get_xlim()[1]-ax.get_xlim()[0])/150), int((ax.get_ylim()[1]-ax.get_ylim()[0])/130)),
               extent=(*ax.get_xlim(), *ax.get_ylim()), mincnt=1, cmap=cmap, bins=None, alpha=0.9, lw=0, zorder=6)
counts = hb.get_array(); edges = [1, 3, 6, 11, 21, max(22, int(counts.max()) + 1)]
hb.set_norm(mpl.colors.BoundaryNorm(edges, cmap.N))
boundary(ax, V['версия 2'])
for k in ('версия 3', 'версия 4'):
    gpd.GeoSeries([V[k]], crs=CRS).boundary.plot(ax=ax, color=T['pigment-ink'], lw=1, zorder=11)
h = [mpl.patches.Patch(color=c, label=f'{a}–{b-1}') for c, a, b in zip(T['map-seq'], edges[:-1], edges[1:])]
h.append(Line2D([], [], color=T['map-boundary'], lw=3, linestyle=(0,(6,4)), label='МО Коломна'))
h.append(Line2D([], [], color=T['pigment-ink'], lw=1, label='версии 3, 4'))
legend(ax, h, title='ОБЪЕКТОВ НА ШЕСТИУГОЛЬНИК 150 М')
scalebar(ax); north(ax)
source(ax, 'Магазины, кафе, бары, рестораны, гостиницы, хостелы · ' + SRC_OSM)
save(fig, '03_kolomna_service_density')

## Карта 4. Возраст застройки («Карта возраста домов»)

Нужен файл `kolomna_data/hoith_kolomna_periods.gpkg` из шага 5 ноутбука 01.

In [ ]:
fn = os.path.join(DATA, 'hoith_kolomna_periods.gpkg')
if os.path.exists(fn):
    hh = gpd.read_file(fn).to_crs(CRS)
    periods = ['до 1800','1800–1849','1850–1879','1880–1917','1918–1955','1956–1991','1992+']
    pal = dict(zip(periods, ['#3d2206','#9a5410','#e28723','#efb877','#d6cfc4', T['canal'], T['brick']]))
    fig, ax = new_map('full', bounds=zoom(V['версия 2'], 350), kicker='II · Когнитивный компонент · застройка',
                      title='Коломна — город XIX века')
    for p in periods:
        s = hh[hh['period'] == p]
        if len(s): s.plot(ax=ax, color=pal[p], lw=0, zorder=6)
    hh[hh['period'].isna()].plot(ax=ax, color=T['map-building'], lw=0, zorder=5)
    boundary(ax, V['версия 2'])
    h = [mpl.patches.Patch(color=pal[p], label=p) for p in periods] + [mpl.patches.Patch(color=T['map-building'], label='нет данных')]
    legend(ax, h, title='ГОД ПОСТРОЙКИ')
    scalebar(ax); north(ax)
    source(ax, 'Данные: проект «Карта возраста домов» (how-old-is-this.house), выгрузка 2023-11-18 · ' + SRC_OSM)
    save(fig, '04_kolomna_building_age')
else:
    print('Сначала выполните шаг 5 в ноутбуке 01')

## Карта 5. Доминанты, проекты и точки наблюдения

Координаты — с карты (проверяйте по OSM/Яндекс Картам); правьте список, если команда выберет другие объекты.

In [ ]:
PTS = pd.DataFrame([
  # name, lon, lat, group
  ('Никольский морской собор', 30.29890, 59.92125, 'доминанта'),
  ('Мариинский театр',        30.29620, 59.92573, 'доминанта'),
  ('Новая сцена Мариинки',     30.29330, 59.92680, 'доминанта'),
  ('Новая Голландия',          30.28950, 59.92960, 'доминанта'),
  ('Большая хоральная синагога', 30.28800, 59.92330, 'доминанта'),
  ('Музей-квартира Блока',     30.27960, 59.92360, 'доминанта'),
  ('Больница на Пряжке',       30.27890, 59.92540, 'доминанта'),
  ('Съезжий дом, пл. Репина',  30.27380, 59.91700, 'доминанта'),
  ('Вестибюль «Театральной» (пл. Темирканова)', 30.29000, 59.92555, 'проект'),
  ('ЖК «Остров первых»',       30.27500, 59.92380, 'проект'),
  ('Концертный комплекс, Ново-Адмиралтейский о.', 30.27900, 59.92960, 'проект'),
], columns=['name','lon','lat','group'])
pts = gpd.GeoDataFrame(PTS, geometry=gpd.points_from_xy(PTS.lon, PTS.lat), crs=4326).to_crs(CRS)
fig, ax = new_map('full', bounds=zoom(V['версия 3'], 300), kicker='II · Доминанты и перемены',
                  title='Символы на краях, перемены — на западе')
boundary(ax, V['версия 2'])
d = pts[pts.group=='доминанта']; p = pts[pts.group=='проект']
d.plot(ax=ax, color=T['map-focus'], edgecolor=T['pigment-ink'], linewidth=2, markersize=110, zorder=15)
p.plot(ax=ax, color=T['pigment-paper'], edgecolor=T['pigment-ink'], linewidth=2, marker='s', markersize=110, zorder=15)
for r in pts.itertuples():
    ax.annotate(r.name, (r.geometry.x, r.geometry.y), xytext=(8, 6), textcoords='offset points',
                fontproperties=FP['PT Sans'], fontsize=9, color=T['text'], zorder=16,
                bbox=dict(facecolor=T['ground'], edgecolor='none', pad=1.5))
h = [Line2D([], [], marker='o', ls='', markerfacecolor=T['map-focus'], markeredgecolor=T['pigment-ink'], markeredgewidth=2, markersize=10, label='доминанта, символ'),
     Line2D([], [], marker='s', ls='', markerfacecolor=T['pigment-paper'], markeredgecolor=T['pigment-ink'], markeredgewidth=2, markersize=10, label='планируемый проект'),
     Line2D([], [], color=T['map-boundary'], lw=3, linestyle=(0,(6,4)), label='МО Коломна')]
legend(ax, h)
scalebar(ax); north(ax)
source(ax, 'Проекты: ДП, Фонтанка, Коммерсантъ (2025–2026) · ' + SRC_OSM)
save(fig, '05_kolomna_dominants_projects')

## Карта 6. Ментальные карты: где для респондентов начинается Коломна

После полевого этапа: каждый респондент обводит «свою» Коломну в [geojson.io](https://geojson.io) (или вы переносите его рисунок), сохраняете полигоны в `kolomna_data/mental_maps.geojson` с полем `group` (`житель` / `нерезидент`). Карта показывает долю респондентов, включивших каждую точку.

In [ ]:
fn = os.path.join(DATA, 'mental_maps.geojson')
if os.path.exists(fn):
    mm = gpd.read_file(fn).to_crs(CRS)
    fig, ax = new_map('full', bounds=zoom(V['версия 4'], 300), kicker='I · Территория · ментальные карты',
                      title='Где начинается Коломна')
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim(); step = 25
    gx, gy = np.meshgrid(np.arange(x0, x1, step), np.arange(y0, y1, step))
    grid = gpd.GeoSeries(gpd.points_from_xy(gx.ravel(), gy.ravel()), crs=CRS)
    share = np.zeros(len(grid))
    for g in mm.geometry: share += grid.within(g).values
    share = (share / len(mm)).reshape(gx.shape)
    levels = [0.01, 0.2, 0.4, 0.6, 0.8, 1.01]
    ax.contourf(gx, gy, share, levels=levels, colors=T['map-seq'], alpha=0.8, zorder=6)
    boundary(ax, V['версия 2'])
    h = [mpl.patches.Patch(color=c, label=f'{int(a*100)}–{int(min(b,1)*100)} %') for c, a, b in zip(T['map-seq'], levels[:-1], levels[1:])]
    h.append(Line2D([], [], color=T['map-boundary'], lw=3, linestyle=(0,(6,4)), label='МО Коломна'))
    legend(ax, h, title=f'ДОЛЯ РЕСПОНДЕНТОВ (N = {len(mm)})')
    scalebar(ax); north(ax)
    source(ax, 'Ментальные карты: полевой опрос команды, 2026 · ' + SRC_OSM)
    save(fig, '06_kolomna_mental_maps')
else:
    print('Нет mental_maps.geojson — карта появится после полевого этапа')

## Карта 7. Маршрут go-along / точки наблюдения (шаблон)

Положите трек в `kolomna_data/route.geojson` (LineString) — например, экспорт из Яндекс Карт или Strava.

In [ ]:
fn = os.path.join(DATA, 'route.geojson')
if os.path.exists(fn):
    rt = gpd.read_file(fn).to_crs(CRS)
    fig, ax = new_map('half', bounds=zoom(rt.union_all(), 250), kicker='II · Практический компонент · go-along',
                      title='Маршрут жителя')
    boundary(ax, V['версия 2'])
    rt.plot(ax=ax, color=T['map-route'], lw=4, zorder=14)
    legend(ax, [Line2D([], [], color=T['map-route'], lw=4, label='маршрут go-along'),
                Line2D([], [], color=T['map-boundary'], lw=3, linestyle=(0,(6,4)), label='МО Коломна')])
    scalebar(ax, 200); north(ax)
    source(ax, 'Трек: полевое исследование команды, 2026 · ' + SRC_OSM)
    save(fig, '07_kolomna_route')
else:
    print('Нет route.geojson')